# EuroSAT RGB + Multispectral + SAR Paired TFRecord Creation

This notebook:

- reads EuroSAT optical RGB, multispectral, and SAR folders
- pairs all three modalities using class name and numeric file ID
- validates RGB, 13-band multispectral, and 2-band SAR tensors
- creates one shared stratified train/validation/test split
- computes training-only normalization statistics for multispectral and SAR data
- writes sharded paired TFRecords
- saves metadata CSV files and a manifest
- validates the generated TFRecords
- uploads the completed TFRecord directory to Kaggle

### Stored tensor formats

- RGB: `64 × 64 × 3`, `uint8`
- Multispectral: `64 × 64 × 13`, `float16`
- SAR: `64 × 64 × 2`, `float16`
- SAR band order: `VV, VH`


In [1]:
!pip install -q -U rasterio kagglehub

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.1/40.1 kB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.6/70.6 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.0/231.0 kB 8.1 MB/s eta 0:00:00


## 1. Imports and configuration

In [2]:
import gc
import json
import os
import random
import re
import shutil
from pathlib import Path

import kagglehub
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rasterio
import tensorflow as tf
from kaggle_secrets import UserSecretsClient
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

ROOT_DIR = Path("/kaggle/input/datasets/glitchr/eurodata")

RGB_DIR = ROOT_DIR / "EuroSAT_RGB" / "EuroSAT_RGB"
MS_DIR = ROOT_DIR / "EuroSAT_MS" / "EuroSAT_MS"
SAR_DIR = ROOT_DIR / "EuroSAT-SAR" / "EuroSAT-SAR"

OUTPUT_ROOT = Path("/kaggle/working")
TFREC_DIR = OUTPUT_ROOT / "eurosat-rgb-ms-sar-paired-tfrecords"

IMAGE_SIZE = 64
RGB_CHANNELS = 3
MS_CHANNELS = 13
SAR_CHANNELS = 2

VALIDATION_SIZE = 0.15
TEST_SIZE = 0.15

TRAIN_SHARDS = 37
VALIDATION_SHARDS = 8
TEST_SHARDS = 8

KAGGLE_DATASET_SLUG = "eurosat-rgb-ms-sar-paired-tfrecords"
UPLOAD_TO_KAGGLE = True
OVERWRITE_OUTPUT_DIR = True

CLASS_NAMES = [
    "AnnualCrop",
    "Forest",
    "HerbaceousVegetation",
    "Highway",
    "Industrial",
    "Pasture",
    "PermanentCrop",
    "Residential",
    "River",
    "SeaLake",
]

CLASS_TO_INDEX = {
    label: index
    for index, label in enumerate(CLASS_NAMES)
}

print("TensorFlow:", tf.__version__)
print("RGB directory:", RGB_DIR)
print("MS directory:", MS_DIR)
print("SAR directory:", SAR_DIR)
print("Output directory:", TFREC_DIR)


2026-06-24 22:03:38.526893: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1782338618.799399      58 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1782338618.878756      58 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1782338619.514809      58 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1782338619.514910      58 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1782338619.514915      58 computation_placer.cc:177] computation placer alr

TensorFlow: 2.19.0
RGB directory: /kaggle/input/datasets/glitchr/eurodata/EuroSAT_RGB/EuroSAT_RGB
MS directory: /kaggle/input/datasets/glitchr/eurodata/EuroSAT_MS/EuroSAT_MS
SAR directory: /kaggle/input/datasets/glitchr/eurodata/EuroSAT-SAR/EuroSAT-SAR
Output directory: /kaggle/working/eurosat-rgb-ms-sar-paired-tfrecords


## 2. Prepare the output directory

In [3]:
if OVERWRITE_OUTPUT_DIR and TFREC_DIR.exists():
    shutil.rmtree(TFREC_DIR)
TFREC_DIR.mkdir(parents=True, exist_ok=True)
print("Prepared output directory:", TFREC_DIR)

Prepared output directory: /kaggle/working/eurosat-rgb-ms-sar-paired-tfrecords


## 3. Discover and pair the three modalities

In [4]:
RGB_EXTENSIONS = {".jpg", ".jpeg", ".png"}
SAR_EXTENSIONS = MS_EXTENSIONS = {".tif", ".tiff"}

def extract_file_id(path):
    numbers = re.findall(r"\d+", Path(path).stem)

    if not numbers:
        return None

    return int(numbers[-1])

def collect_modality_files(root, extensions, path_column):
    root = Path(root)
    rows = []

    for label in CLASS_NAMES:
        class_dir = root / label

        if not class_dir.exists():
            raise FileNotFoundError(f"Missing class directory: {class_dir}")

        for path in sorted(class_dir.rglob("*")):
            if path.is_file() and path.suffix.lower() in extensions:
                rows.append({
                    "file_id": f"{label}_{extract_file_id(path)}",
                    "label": label,
                    "target": CLASS_TO_INDEX[label],
                    path_column: str(path),
                })

    dataframe = pd.DataFrame(rows)

    if dataframe.empty:
        raise RuntimeError(f"No files were found under: {root}")

    dataframe = (
        dataframe
        .dropna(subset=["file_id"])
        .drop_duplicates(subset=["label", "file_id"], keep="first")
        .sort_values(["label", "file_id"])
        .reset_index(drop=True)
    )

    return dataframe

rgb_df = collect_modality_files(RGB_DIR, RGB_EXTENSIONS, "rgb_path")
ms_df = collect_modality_files(MS_DIR, MS_EXTENSIONS, "ms_path")
sar_df = collect_modality_files(SAR_DIR, SAR_EXTENSIONS, "sar_path")

print("RGB files:", len(rgb_df))
print("MS files:", len(ms_df))
print("SAR files:", len(sar_df))

RGB files: 27000
MS files: 27000
SAR files: 27000


In [5]:
rgb_df

,file_id,label,target,rgb_path
0,AnnualCrop_1,AnnualCrop,0,/kaggle/input/datasets/glitchr/eurodata/EuroSA...
1,AnnualCrop_10,AnnualCrop,0,/kaggle/input/datasets/glitchr/eurodata/EuroSA...
2,AnnualCrop_100,AnnualCrop,0,/kaggle/input/datasets/glitchr/eurodata/EuroSA...
3,AnnualCrop_1000,AnnualCrop,0,/kaggle/input/datasets/glitchr/eurodata/EuroSA...
4,AnnualCrop_1001,AnnualCrop,0,/kaggle/input/datasets/glitchr/eurodata/EuroSA...
...,...,...,...,...
26995,SeaLake_995,SeaLake,9,/kaggle/input/datasets/glitchr/eurodata/EuroSA...
26996,SeaLake_996,SeaLake,9,/kaggle/input/datasets/glitchr/eurodata/EuroSA...
26997,SeaLake_997,SeaLake,9,/kaggle/input/datasets/glitchr/eurodata/EuroSA...
26998,SeaLake_998,SeaLake,9,/kaggle/input/datasets/glitchr/eurodata/EuroSA...


In [6]:
paired_df = (rgb_df.merge(ms_df[["file_id","label","ms_path",]],on=["file_id", "label"],how="inner",validate="one_to_one",)
    .merge(sar_df[["file_id","label","sar_path",]],on=["file_id", "label"],how="inner",validate="one_to_one",)
    .sort_values(["label", "file_id"]).reset_index(drop=True))

if paired_df.empty:
    raise RuntimeError("No RGB-MS-SAR triplets were found.")

print("Paired triplets:", len(paired_df))

display(paired_df.head())
display(paired_df["label"].value_counts().reindex(CLASS_NAMES).rename("paired_count").to_frame())

Paired triplets: 27000


,file_id,label,target,rgb_path,ms_path,sar_path
0,AnnualCrop_1,AnnualCrop,0,/kaggle/input/datasets/glitchr/eurodata/EuroSA...,/kaggle/input/datasets/glitchr/eurodata/EuroSA...,/kaggle/input/datasets/glitchr/eurodata/EuroSA...
1,AnnualCrop_10,AnnualCrop,0,/kaggle/input/datasets/glitchr/eurodata/EuroSA...,/kaggle/input/datasets/glitchr/eurodata/EuroSA...,/kaggle/input/datasets/glitchr/eurodata/EuroSA...
2,AnnualCrop_100,AnnualCrop,0,/kaggle/input/datasets/glitchr/eurodata/EuroSA...,/kaggle/input/datasets/glitchr/eurodata/EuroSA...,/kaggle/input/datasets/glitchr/eurodata/EuroSA...
3,AnnualCrop_1000,AnnualCrop,0,/kaggle/input/datasets/glitchr/eurodata/EuroSA...,/kaggle/input/datasets/glitchr/eurodata/EuroSA...,/kaggle/input/datasets/glitchr/eurodata/EuroSA...
4,AnnualCrop_1001,AnnualCrop,0,/kaggle/input/datasets/glitchr/eurodata/EuroSA...,/kaggle/input/datasets/glitchr/eurodata/EuroSA...,/kaggle/input/datasets/glitchr/eurodata/EuroSA...


,paired_count
label,
AnnualCrop,3000
Forest,3000
HerbaceousVegetation,3000
Highway,2500
Industrial,2500
Pasture,2000
PermanentCrop,2500
Residential,3000
River,2500


In [7]:
paired_df.sample().to_dict()

{'file_id': {5312: 'Forest_38'},
 'label': {5312: 'Forest'},
 'target': {5312: 1},
 'rgb_path': {5312: '/kaggle/input/datasets/glitchr/eurodata/EuroSAT_RGB/EuroSAT_RGB/Forest/Forest_38.jpg'},
 'ms_path': {5312: '/kaggle/input/datasets/glitchr/eurodata/EuroSAT_MS/EuroSAT_MS/Forest/Forest_38.tif'},
 'sar_path': {5312: '/kaggle/input/datasets/glitchr/eurodata/EuroSAT-SAR/EuroSAT-SAR/Forest/Forest_38.tif'}}

## 4. Validate source tensor shapes and dtypes

In [8]:
def read_rgb(path):
    raw = tf.io.read_file(str(path))
    image = tf.io.decode_image(raw,channels=RGB_CHANNELS,expand_animations=False,)
    image = tf.image.resize(image,[IMAGE_SIZE, IMAGE_SIZE],antialias=True,)
    image = tf.cast(tf.clip_by_value(image, 0, 255), tf.uint8,)
    return image.numpy()

def read_raster(path,expected_channels,):
    with rasterio.open(path) as dataset:
        array = dataset.read().astype(np.float32)
    if array.ndim != 3:
        raise ValueError(f"Expected a 3D raster, received {array.shape}: {path}")
    if array.shape[0] < expected_channels:
        raise ValueError(f"Expected at least {expected_channels} bands, received {array.shape[0]}: {path}")
    array = array[:expected_channels]
    array = np.moveaxis(array, 0, -1)
    if array.shape[:2] != (IMAGE_SIZE,IMAGE_SIZE,):
        array = tf.image.resize(array,[IMAGE_SIZE, IMAGE_SIZE],method="bilinear",).numpy()
    array = np.nan_to_num(array,nan=0.0,posinf=0.0,neginf=0.0,)
    return array.astype(np.float32)

def read_ms(path):
    return read_raster(path,MS_CHANNELS,)

def read_sar(path):
    return read_raster(path,SAR_CHANNELS,)

sample_row = paired_df.iloc[0]
sample_rgb = read_rgb(sample_row["rgb_path"])
sample_ms = read_ms(sample_row["ms_path"])
sample_sar = read_sar(sample_row["sar_path"])

print("RGB:", sample_rgb.shape, sample_rgb.dtype)
print("MS:", sample_ms.shape, sample_ms.dtype)
print("SAR:", sample_sar.shape, sample_sar.dtype)

assert sample_rgb.shape == (IMAGE_SIZE,IMAGE_SIZE,RGB_CHANNELS,)
assert sample_ms.shape == (IMAGE_SIZE,IMAGE_SIZE,MS_CHANNELS,)
assert sample_sar.shape == (IMAGE_SIZE,IMAGE_SIZE,SAR_CHANNELS,)

2026-06-24 22:06:21.145830: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


RGB: (64, 64, 3) uint8
MS: (64, 64, 13) float32
SAR: (64, 64, 2) float32


## 5. Optional visual validation

In [ ]:
def percentile_stretch(channel,lower=2.0,upper=98.0,):
    channel = np.asarray(channel,dtype=np.float32,)

    finite = np.isfinite(channel)

    if not finite.any():
        return np.zeros_like(channel)

    low = np.percentile(channel[finite],lower,)

    high = np.percentile(channel[finite],upper,)

    if high <= low:
        return np.zeros_like(channel)

    channel = np.nan_to_num(channel,nan=low,posinf=high,neginf=low,)

    return np.clip((channel - low) / (high - low),0.0,1.0,)

def ms_gray_preview(ms):
    visible_mean = np.mean(ms[..., :3],axis=-1,)

    return percentile_stretch(visible_mean)

def sar_gray_preview(sar):
    vv = percentile_stretch(sar[..., 0])

    vh = percentile_stretch(sar[..., 1])

    return 0.5 * (vv + vh)

sample_rows = paired_df.groupby("label",group_keys=False,).sample(n=1,random_state=SEED,).reset_index(drop=True)

figure, axes = plt.subplots(len(sample_rows),3,figsize=(9,3 * len(sample_rows),),)

for row_index, row in enumerate(
    sample_rows.itertuples(index=False)
):
    rgb = read_rgb(row.rgb_path)
    ms = read_ms(row.ms_path)
    sar = read_sar(row.sar_path)

    axes[row_index, 0].imshow(rgb)
    axes[row_index, 0].set_title(f"RGB - {row.label}")

    axes[row_index, 1].imshow(ms_gray_preview(ms),cmap="gray",vmin=0,vmax=1,)
    axes[row_index, 1].set_title(f"MS grayscale - {row.label}")

    axes[row_index, 2].imshow(sar_gray_preview(sar),cmap="gray",vmin=0,vmax=1,)
    axes[row_index, 2].set_title(f"SAR grayscale - {row.label}")

    for axis in axes[row_index]:
        axis.axis("off")

plt.tight_layout()
plt.show()

## 6. Create one shared stratified split

In [10]:
train_df, temporary_df = train_test_split(
    paired_df,
    test_size=VALIDATION_SIZE + TEST_SIZE,
    random_state=SEED,
    stratify=paired_df["target"],
)

relative_test_size = TEST_SIZE/ (VALIDATION_SIZE + TEST_SIZE)

validation_df, test_df = train_test_split(
    temporary_df,
    test_size=relative_test_size,
    random_state=SEED,
    stratify=temporary_df["target"],
)

train_df = train_df.reset_index(drop=True)
validation_df = validation_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

split_summary = pd.DataFrame({
    "split": [
        "train",
        "validation",
        "test",
    ],
    "examples": [len(train_df),len(validation_df),len(test_df)],})

display(split_summary)
assert set(train_df["file_id"]).isdisjoint(validation_df["file_id"])
assert set(train_df["file_id"]).isdisjoint(test_df["file_id"])
assert set(validation_df["file_id"]).isdisjoint(test_df["file_id"])

,split,examples
0,train,18900
1,validation,4050
2,test,4050


In [11]:
paired_df

,file_id,label,target,rgb_path,ms_path,sar_path
0,AnnualCrop_1,AnnualCrop,0,/kaggle/input/datasets/glitchr/eurodata/EuroSA...,/kaggle/input/datasets/glitchr/eurodata/EuroSA...,/kaggle/input/datasets/glitchr/eurodata/EuroSA...
1,AnnualCrop_10,AnnualCrop,0,/kaggle/input/datasets/glitchr/eurodata/EuroSA...,/kaggle/input/datasets/glitchr/eurodata/EuroSA...,/kaggle/input/datasets/glitchr/eurodata/EuroSA...
2,AnnualCrop_100,AnnualCrop,0,/kaggle/input/datasets/glitchr/eurodata/EuroSA...,/kaggle/input/datasets/glitchr/eurodata/EuroSA...,/kaggle/input/datasets/glitchr/eurodata/EuroSA...
3,AnnualCrop_1000,AnnualCrop,0,/kaggle/input/datasets/glitchr/eurodata/EuroSA...,/kaggle/input/datasets/glitchr/eurodata/EuroSA...,/kaggle/input/datasets/glitchr/eurodata/EuroSA...
4,AnnualCrop_1001,AnnualCrop,0,/kaggle/input/datasets/glitchr/eurodata/EuroSA...,/kaggle/input/datasets/glitchr/eurodata/EuroSA...,/kaggle/input/datasets/glitchr/eurodata/EuroSA...
...,...,...,...,...,...,...
26995,SeaLake_995,SeaLake,9,/kaggle/input/datasets/glitchr/eurodata/EuroSA...,/kaggle/input/datasets/glitchr/eurodata/EuroSA...,/kaggle/input/datasets/glitchr/eurodata/EuroSA...
26996,SeaLake_996,SeaLake,9,/kaggle/input/datasets/glitchr/eurodata/EuroSA...,/kaggle/input/datasets/glitchr/eurodata/EuroSA...,/kaggle/input/datasets/glitchr/eurodata/EuroSA...
26997,SeaLake_997,SeaLake,9,/kaggle/input/datasets/glitchr/eurodata/EuroSA...,/kaggle/input/datasets/glitchr/eurodata/EuroSA...,/kaggle/input/datasets/glitchr/eurodata/EuroSA...
26998,SeaLake_998,SeaLake,9,/kaggle/input/datasets/glitchr/eurodata/EuroSA...,/kaggle/input/datasets/glitchr/eurodata/EuroSA...,/kaggle/input/datasets/glitchr/eurodata/EuroSA...


## 7. Compute training-only MS and SAR normalization statistics

In [12]:
def streaming_band_statistics(paths, reader, channels, description):
    pixel_count = 0

    channel_sum = np.zeros(channels, dtype=np.float64)
    channel_squared_sum = np.zeros(channels, dtype=np.float64)

    for path in tqdm(paths, desc=description):
        image = reader(path)

        flattened = image.reshape(-1, channels).astype(np.float64)

        channel_sum += flattened.sum(axis=0)
        channel_squared_sum += np.square(flattened).sum(axis=0)

        pixel_count += len(flattened)

    mean = channel_sum / max(pixel_count, 1)

    variance = (
        channel_squared_sum / max(pixel_count, 1)
        - np.square(mean)
    )

    std = np.sqrt(np.maximum(variance, 1e-8))

    return (
        mean.astype(np.float32),
        std.astype(np.float32),
    )


MS_MEAN, MS_STD = streaming_band_statistics(
    train_df["ms_path"],
    read_ms,
    MS_CHANNELS,
    "Computing multispectral statistics",
)

SAR_MEAN, SAR_STD = streaming_band_statistics(
    train_df["sar_path"],
    read_sar,
    SAR_CHANNELS,
    "Computing SAR statistics",
)

print("MS mean:", MS_MEAN)
print("MS std:", MS_STD)
print("SAR band order: [VV, VH]")
print("SAR mean:", SAR_MEAN)
print("SAR std:", SAR_STD)

assert np.all(np.isfinite(MS_MEAN))
assert np.all(np.isfinite(MS_STD))
assert np.all(MS_STD > 0)

assert np.all(np.isfinite(SAR_MEAN))
assert np.all(np.isfinite(SAR_STD))
assert np.all(SAR_STD > 0)

Computing multispectral statistics:   0%|          | 0/18900 [00:00<?, ?it/s]

Computing SAR statistics:   0%|          | 0/18900 [00:00<?, ?it/s]

MS mean: [1353.2472    1116.5975    1040.9788     945.2394    1197.3502
 2000.3962    2371.259     2298.4998     730.8872      12.0971575
 1819.6499    1116.7406    2596.9368   ]
MS std: [ 245.02089    332.93954    394.64838    593.8269     566.4712
  860.3379    1086.4225    1118.0255     403.62488      4.7438297
 1003.40875    759.8051    1231.3096   ]
SAR band order: [VV, VH]
SAR mean: [-11.702763 -18.562027]
SAR std: [6.1591363 6.29818  ]


## 8. TFRecord serialization helpers

In [17]:
def bytes_feature(value):
    if isinstance(value, str):
        value = value.encode("utf-8")
    return tf.train.Feature(bytes_list=tf.train.BytesList(value=[value]))

def int_feature(value):
    return tf.train.Feature(int64_list=tf.train.Int64List(value=[int(value)]))

def serialize_triplet(row):
    rgb = read_rgb(row.rgb_path)
    ms = read_ms(row.ms_path).astype(np.float16)
    sar = read_sar(row.sar_path).astype(np.float16)

    example = tf.train.Example(
        features=tf.train.Features(
            feature={
                "rgb_raw": bytes_feature(tf.io.serialize_tensor(rgb).numpy()),
                "ms_raw": bytes_feature(tf.io.serialize_tensor(ms).numpy()),
                "sar_raw": bytes_feature(tf.io.serialize_tensor(sar).numpy()),
                "target": int_feature(row.target),
                "file_id": bytes_feature(row.file_id),
                "label": bytes_feature(row.label),
            }
        )
    )

    return example.SerializeToString()

## 9. Write sharded TFRecords

In [18]:
def write_sharded_tfrecords(dataframe, split, shard_count):
    shard_count = min(shard_count, len(dataframe))

    index_groups = np.array_split(
        np.arange(len(dataframe)),
        shard_count,
    )

    shard_names = []

    for shard_index, indices in enumerate(
        tqdm(index_groups, desc=f"Writing {split}")
    ):
        shard_name = f"{split}-{shard_index:05d}.tfrecord"
        shard_path = TFREC_DIR / shard_name

        with tf.io.TFRecordWriter(str(shard_path)) as writer:
            for index in indices:
                row = dataframe.iloc[int(index)]
                writer.write(serialize_triplet(row))

        shard_names.append(shard_name)

    return shard_names


split_frames = {
    "train": train_df,
    "validation": validation_df,
    "test": test_df,
}

split_shard_counts = {
    "train": TRAIN_SHARDS,
    "validation": VALIDATION_SHARDS,
    "test": TEST_SHARDS,
}

manifest = {
    "dataset": "EuroSAT paired RGB, 13-band multispectral and dual-polarization SAR",
    "image_size": IMAGE_SIZE,
    "rgb_channels": RGB_CHANNELS,
    "ms_channels": MS_CHANNELS,
    "sar_channels": SAR_CHANNELS,
    "sar_band_order": ["VV", "VH"],
    "ms_normalization": {
        "mean": MS_MEAN.tolist(),
        "std": MS_STD.tolist(),
    },
    "sar_normalization": {
        "mean": SAR_MEAN.tolist(),
        "std": SAR_STD.tolist(),
    },
    "splits": {},
}

for split, dataframe in split_frames.items():
    shards = write_sharded_tfrecords(
        dataframe,
        split,
        split_shard_counts[split],
    )

    manifest["splits"][split] = {
        "examples": len(dataframe),
        "shards": shards,
        "compression_type": "",
    }

    dataframe.to_csv(
        TFREC_DIR / f"{split}_metadata.csv",
        index=False,
    )

manifest_path = TFREC_DIR / "manifest.json"

manifest_path.write_text(
    json.dumps(
        manifest,
        indent=2,
    )
)

print("Created TFRecords:", TFREC_DIR)
print("Manifest:", manifest_path)

Writing train:   0%|          | 0/37 [00:00<?, ?it/s]

Writing validation:   0%|          | 0/8 [00:00<?, ?it/s]

Writing test:   0%|          | 0/8 [00:00<?, ?it/s]

Created TFRecords: /kaggle/working/eurosat-rgb-ms-sar-paired-tfrecords
Manifest: /kaggle/working/eurosat-rgb-ms-sar-paired-tfrecords/manifest.json


## 10. Validate the generated dataset

In [19]:
FEATURE_SPEC = {
    "rgb_raw": tf.io.FixedLenFeature([], tf.string),
    "ms_raw": tf.io.FixedLenFeature([], tf.string),
    "sar_raw": tf.io.FixedLenFeature([], tf.string),
    "target": tf.io.FixedLenFeature([], tf.int64),
    "file_id": tf.io.FixedLenFeature([], tf.string),
    "label": tf.io.FixedLenFeature([], tf.string),
}

def decode_validation_record(serialized):
    item = tf.io.parse_single_example(
        serialized,
        FEATURE_SPEC,
    )

    rgb = tf.io.parse_tensor(item["rgb_raw"], out_type=tf.uint8)
    ms = tf.io.parse_tensor(item["ms_raw"], out_type=tf.float16)
    sar = tf.io.parse_tensor(item["sar_raw"], out_type=tf.float16)

    rgb = tf.ensure_shape(
        rgb,
        [IMAGE_SIZE, IMAGE_SIZE, RGB_CHANNELS],
    )

    ms = tf.ensure_shape(
        ms,
        [IMAGE_SIZE, IMAGE_SIZE, MS_CHANNELS],
    )

    sar = tf.ensure_shape(
        sar,
        [IMAGE_SIZE, IMAGE_SIZE, SAR_CHANNELS],
    )

    return {
        "rgb": rgb,
        "ms": ms,
        "sar": sar,
        "target": item["target"],
        "file_id": item["file_id"],
        "label": item["label"],
    }


validation_rows = []

for split in ("train", "validation", "test"):
    shard_paths = [
        str(TFREC_DIR / shard_name)
        for shard_name in manifest["splits"][split]["shards"]
    ]

    dataset = tf.data.TFRecordDataset(
        shard_paths,
        compression_type="",
        num_parallel_reads=tf.data.AUTOTUNE,
    )

    count = dataset.reduce(
        tf.constant(0, tf.int64),
        lambda total, _: total + 1,
    )

    count = int(count.numpy())
    expected = int(manifest["splits"][split]["examples"])

    validation_rows.append({
        "split": split,
        "expected": expected,
        "actual": count,
        "valid": count == expected,
    })

    if count != expected:
        raise RuntimeError(
            f"{split}: expected {expected}, found {count}"
        )

display(pd.DataFrame(validation_rows))

sample_dataset = tf.data.TFRecordDataset([
    str(TFREC_DIR / manifest["splits"]["train"]["shards"][0])
])

sample = decode_validation_record(
    next(iter(sample_dataset))
)

print("Decoded RGB:", sample["rgb"].shape, sample["rgb"].dtype)
print("Decoded MS:", sample["ms"].shape, sample["ms"].dtype)
print("Decoded SAR:", sample["sar"].shape, sample["sar"].dtype)
print("Target:", int(sample["target"].numpy()))
print("File ID:", sample["file_id"].numpy().decode("utf-8"))
print("Label:", sample["label"].numpy().decode("utf-8"))

,split,expected,actual,valid
0,train,18900,18900,True
1,validation,4050,4050,True
2,test,4050,4050,True


Decoded RGB: (64, 64, 3) <dtype: 'uint8'>
Decoded MS: (64, 64, 13) <dtype: 'float16'>
Decoded SAR: (64, 64, 2) <dtype: 'float16'>
Target: 7
File ID: Residential_238
Label: Residential


## 11. Inspect generated files

In [20]:
generated_files = []

for path in sorted(TFREC_DIR.rglob("*")):
    if path.is_file():
        generated_files.append({"file": path.name, "size_mb": round(path.stat().st_size / 1024**2, 3)})

generated_files_df = pd.DataFrame(generated_files)

display(generated_files_df)

print("Total size:", round(generated_files_df["size_mb"].sum(), 2), "MB")

,file,size_mb
0,manifest.json,0.003
1,test-00000.tfrecord,65.463
2,test-00001.tfrecord,65.463
3,test-00002.tfrecord,65.334
4,test-00003.tfrecord,65.334
5,test-00004.tfrecord,65.334
6,test-00005.tfrecord,65.334
7,test-00006.tfrecord,65.334
8,test-00007.tfrecord,65.334
9,test_metadata.csv,1.191


Total size: 3494.13 MB


## 12. Upload the complete TFRecord directory to Kaggle

In [23]:
user_secrets = UserSecretsClient()

os.environ["KAGGLE_USERNAME"] = user_secrets.get_secret("KAGGLE_USERNAME")
os.environ["KAGGLE_KEY"] = user_secrets.get_secret("KAGGLE_KEY")

KAGGLE_USERNAME = os.environ["KAGGLE_USERNAME"].strip()

KAGGLE_DATASET_HANDLE = f"{KAGGLE_USERNAME}/{KAGGLE_DATASET_SLUG}"

print("Kaggle dataset handle:", KAGGLE_DATASET_HANDLE)

Kaggle dataset handle: glitchr/eurosat-rgb-ms-sar-paired-tfrecords


In [24]:
UPLOAD_SUCCEEDED = False
UPLOAD_ERROR = ""

if UPLOAD_TO_KAGGLE:
    try:
        kagglehub.dataset_upload(KAGGLE_DATASET_HANDLE, TFREC_DIR, ignore_patterns=["*.tmp", "*.log"])
        UPLOAD_SUCCEEDED = True
        print("Uploaded dataset:", KAGGLE_DATASET_HANDLE)

    except Exception as error:
        UPLOAD_ERROR = f"{type(error).__name__}: {error}"
        print("Kaggle upload failed:")
        print(UPLOAD_ERROR)
else:
    print("Upload skipped because UPLOAD_TO_KAGGLE=False")

Uploading Dataset https://kaggle.com/datasets/glitchr/eurosat-rgb-ms-sar-paired-tfrecords ...
More than 50 files detected, creating a zip archive...
Starting upload for file /tmp/tmpf8zbn6wa/archive.zip


Uploading: 100%|██████████| 3.66G/3.66G [00:58<00:00, 62.6MB/s]

Upload successful: /tmp/tmpf8zbn6wa/archive.zip (3GB)


Your dataset has been created.
Files are being processed...
See at: https://kaggle.com/datasets/glitchr/eurosat-rgb-ms-sar-paired-tfrecords
Uploaded dataset: glitchr/eurosat-rgb-ms-sar-paired-tfrecords


## 13. Final summary

In [25]:
summary = {
    "dataset_handle": KAGGLE_DATASET_HANDLE,
    "local_directory": str(TFREC_DIR),
    "upload_succeeded": UPLOAD_SUCCEEDED,
    "upload_error": UPLOAD_ERROR,
    "modalities": ["RGB", "Multispectral", "SAR"],
    "rgb_shape": [IMAGE_SIZE, IMAGE_SIZE, RGB_CHANNELS],
    "ms_shape": [IMAGE_SIZE, IMAGE_SIZE, MS_CHANNELS],
    "sar_shape": [IMAGE_SIZE, IMAGE_SIZE, SAR_CHANNELS],
    "sar_band_order": ["VV", "VH"],
    "splits": {split: manifest["splits"][split] for split in ("train", "validation", "test")},
}

summary_path = TFREC_DIR / "creation_summary.json"

summary_path.write_text(json.dumps(summary, indent=2))

print(json.dumps(summary, indent=2))

{
  "dataset_handle": "glitchr/eurosat-rgb-ms-sar-paired-tfrecords",
  "local_directory": "/kaggle/working/eurosat-rgb-ms-sar-paired-tfrecords",
  "upload_succeeded": true,
  "upload_error": "",
  "modalities": [
    "RGB",
    "Multispectral",
    "SAR"
  ],
  "rgb_shape": [
    64,
    64,
    3
  ],
  "ms_shape": [
    64,
    64,
    13
  ],
  "sar_shape": [
    64,
    64,
    2
  ],
  "sar_band_order": [
    "VV",
    "VH"
  ],
  "splits": {
    "train": {
      "examples": 18900,
      "shards": [
        "train-00000.tfrecord",
        "train-00001.tfrecord",
        "train-00002.tfrecord",
        "train-00003.tfrecord",
        "train-00004.tfrecord",
        "train-00005.tfrecord",
        "train-00006.tfrecord",
        "train-00007.tfrecord",
        "train-00008.tfrecord",
        "train-00009.tfrecord",
        "train-00010.tfrecord",
        "train-00011.tfrecord",
        "train-00012.tfrecord",
        "train-00013.tfrecord",
        "train-00014.tfrecord",
        "t